In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

currentRoot = Path.cwd().resolve()
projectRoot = currentRoot.parent if currentRoot.name == 'notebooks' else currentRoot
if not (projectRoot / 'notebooks').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory.')

dataRoot = projectRoot / 'data'
rawRoot = dataRoot / 'raw'
interimRoot = dataRoot / 'interim'
processedRoot = dataRoot / 'processed'
figureRoot = projectRoot / 'reports' / 'figures'
for outputRoot in (rawRoot, interimRoot, processedRoot, figureRoot):
    outputRoot.mkdir(parents=True, exist_ok=True)

configuredRawRoot = os.environ.get('SKIN_DATA_ROOT')
rawCandidates = [Path(configuredRawRoot).expanduser()] if configuredRawRoot else []
rawCandidates.extend([rawRoot, projectRoot.parent / 'Data'])

def containsDatasetFiles(path: Path) -> bool:
    datasetSuffixes = {'.csv', '.zip', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    return path.exists() and any(
        candidate.is_file() and candidate.suffix.lower() in datasetSuffixes
        for candidate in path.rglob('*')
    )

sourceDataRoot = next((path.resolve() for path in rawCandidates if containsDatasetFiles(path)), None)
if sourceDataRoot is None:
    raise FileNotFoundError('No raw dataset found. Populate data/raw or set SKIN_DATA_ROOT.')

def toProjectPath(path: Path) -> str:
    return Path(os.path.relpath(path.resolve(), projectRoot)).as_posix()

def fromProjectPath(pathValue: str) -> Path:
    return (projectRoot / Path(pathValue)).resolve()

print(f'Project root: {projectRoot}')
print(f'Raw dataset root: {sourceDataRoot}')

Project root: D:\sunib\Research\Skin\skin-domain-shift
Raw dataset root: D:\sunib\Research\Skin\Data


In [2]:
from sklearn.model_selection import StratifiedGroupKFold

clinicalFiveClassPath = processedRoot / 'clinicalFiveClass.csv'
isicFiveClassPath = processedRoot / 'isicFiveClass.csv'
duplicateGroupsPath = processedRoot / 'duplicateGroups.csv'
for requiredPath in (clinicalFiveClassPath, isicFiveClassPath, duplicateGroupsPath):
    if not requiredPath.exists():
        raise FileNotFoundError(f'Run notebooks 02–04 first; missing {requiredPath.name}.')

clinicalData = pd.read_csv(clinicalFiveClassPath, dtype={'patientId': 'string', 'lesionId': 'string'}, low_memory=False)
isicData = pd.read_csv(isicFiveClassPath, low_memory=False)
duplicateGroups = pd.read_csv(duplicateGroupsPath, low_memory=False)

In [3]:
class UnionFind:
    def __init__(self, items: list[str]):
        self.parent = {item: item for item in items}

    def find(self, item: str) -> str:
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, leftItem: str, rightItem: str) -> None:
        leftRoot = self.find(leftItem)
        rightRoot = self.find(rightItem)
        if leftRoot != rightRoot:
            smallerRoot, largerRoot = sorted([leftRoot, rightRoot])
            self.parent[largerRoot] = smallerRoot

patientTokens = [f'patient:{value}' for value in clinicalData['patientId'].astype(str)]
groupBuilder = UnionFind(sorted(set(patientTokens)))
duplicateByPath = duplicateGroups.set_index('imagePath')['duplicateGroup'].to_dict() if len(duplicateGroups) else {}
clinicalData['confirmedDuplicateGroup'] = clinicalData['imagePath'].map(duplicateByPath)

for duplicateGroup, group in clinicalData.dropna(subset=['confirmedDuplicateGroup']).groupby('confirmedDuplicateGroup'):
    groupPatients = sorted({f'patient:{value}' for value in group['patientId'].astype(str)})
    for patientToken in groupPatients[1:]:
        groupBuilder.union(groupPatients[0], patientToken)

clinicalData['splitGroup'] = [groupBuilder.find(token) for token in patientTokens]
print(f'Patients: {clinicalData.patientId.nunique():,}; connected split groups: {clinicalData.splitGroup.nunique():,}')
print('Minority-class patient counts:')
print(clinicalData.groupby('harmonizedLabel')['patientId'].nunique().sort_values())

Patients: 1,246; connected split groups: 1,246
Minority-class patient counts:
harmonizedLabel
MEL     36
SCC    133
NV     185
BCC    513
AK     525
Name: patientId, dtype: int64


In [4]:
def chooseHoldout(frame: pd.DataFrame, holdoutFraction: float, randomSeed: int) -> tuple[np.ndarray, np.ndarray]:
    splitCount = max(2, round(1 / holdoutFraction))
    splitter = StratifiedGroupKFold(n_splits=splitCount, shuffle=True, random_state=randomSeed)
    overallDistribution = frame['harmonizedLabel'].value_counts(normalize=True)
    bestResult = None
    for trainIndices, holdoutIndices in splitter.split(frame, frame['harmonizedLabel'], frame['splitGroup']):
        holdoutFrame = frame.iloc[holdoutIndices]
        holdoutDistribution = holdoutFrame['harmonizedLabel'].value_counts(normalize=True).reindex(overallDistribution.index, fill_value=0)
        sizeError = abs(len(holdoutFrame) / len(frame) - holdoutFraction)
        distributionError = float((holdoutDistribution - overallDistribution).abs().mean())
        missingClassPenalty = len(set(overallDistribution.index) - set(holdoutFrame['harmonizedLabel']))
        score = missingClassPenalty * 10 + sizeError + distributionError
        candidate = (score, tuple(sorted(holdoutFrame['imageId'].astype(str))), trainIndices, holdoutIndices)
        if bestResult is None or candidate[:2] < bestResult[:2]:
            bestResult = candidate
    return bestResult[2], bestResult[3]

trainValIndices, testIndices = chooseHoldout(clinicalData, 0.15, RANDOM_SEED)
trainValData = clinicalData.iloc[trainValIndices].copy()
clinicalTest = clinicalData.iloc[testIndices].copy()
relativeValFraction = 0.15 / 0.85
trainIndices, valIndices = chooseHoldout(trainValData, relativeValFraction, RANDOM_SEED + 1)
clinicalTrain = trainValData.iloc[trainIndices].copy()
clinicalVal = trainValData.iloc[valIndices].copy()

for splitName, frame in [('train', clinicalTrain), ('val', clinicalVal), ('test', clinicalTest)]:
    frame['split'] = splitName

splitFrames = {'train': clinicalTrain, 'val': clinicalVal, 'test': clinicalTest}
for splitName, frame in splitFrames.items():
    print(f'{splitName}: {len(frame):,} images, {frame.patientId.nunique():,} patients')
    print(frame['harmonizedLabel'].value_counts().sort_index().to_dict())

train: 1,479 images, 889 patients
{'AK': 531, 'BCC': 606, 'MEL': 35, 'NV': 169, 'SCC': 138}
val: 290 images, 180 patients
{'AK': 96, 'BCC': 115, 'MEL': 10, 'NV': 37, 'SCC': 32}
test: 294 images, 177 patients
{'AK': 103, 'BCC': 124, 'MEL': 7, 'NV': 38, 'SCC': 22}


In [5]:
def pairwiseOverlaps(frames: dict[str, pd.DataFrame], columnName: str) -> dict[str, int]:
    overlapCounts = {}
    splitNames = list(frames)
    for leftIndex, leftName in enumerate(splitNames):
        for rightName in splitNames[leftIndex + 1:]:
            leftValues = set(frames[leftName][columnName].dropna().astype(str))
            rightValues = set(frames[rightName][columnName].dropna().astype(str))
            overlapCounts[f'{leftName}-{rightName}'] = len(leftValues & rightValues)
    return overlapCounts

patientOverlap = pairwiseOverlaps(splitFrames, 'patientId')
duplicateOverlap = pairwiseOverlaps(splitFrames, 'confirmedDuplicateGroup')
assert all(count == 0 for count in patientOverlap.values()), patientOverlap
assert all(count == 0 for count in duplicateOverlap.values()), duplicateOverlap
assert sum(len(frame) for frame in splitFrames.values()) == len(clinicalData)
for splitName, frame in splitFrames.items():
    missingClasses = set(clinicalData['harmonizedLabel']) - set(frame['harmonizedLabel'])
    assert not missingClasses, f'{splitName} is missing classes: {sorted(missingClasses)}'

clinicalTrainPath = processedRoot / 'clinicalTrain.csv'
clinicalValPath = processedRoot / 'clinicalVal.csv'
clinicalTestPath = processedRoot / 'clinicalTest.csv'
clinicalTrain.to_csv(clinicalTrainPath, index=False)
clinicalVal.to_csv(clinicalValPath, index=False)
clinicalTest.to_csv(clinicalTestPath, index=False)
print(f'Patient overlaps: {patientOverlap}; exact-duplicate overlaps: {duplicateOverlap}')

Patient overlaps: {'train-val': 0, 'train-test': 0, 'val-test': 0}; exact-duplicate overlaps: {'train-val': 0, 'train-test': 0, 'val-test': 0}


In [6]:
isicLabelsAvailable = bool(len(isicData) > 0 and isicData['harmonizedLabel'].notna().all())
if not isicLabelsAvailable:
    raise RuntimeError('Official ISIC labels are required before supervised domain-shift evaluation.')

isicTrain = isicData.loc[isicData['officialPartition'].eq('officialTrain')].copy()
isicVal = isicData.iloc[0:0].copy()
isicTest = isicData.loc[isicData['officialPartition'].eq('officialTest')].copy()
isicTrain['split'] = 'officialTrainLabelled'
isicTest['split'] = 'officialTestDomainShift'
assert len(isicTrain) > 0 and len(isicTest) > 0
assert set(isicTrain['imageId']).isdisjoint(set(isicTest['imageId']))

isicTrainPath = processedRoot / 'isicTrain.csv'
isicValPath = processedRoot / 'isicVal.csv'
isicTestPath = processedRoot / 'isicTest.csv'
isicTrain.to_csv(isicTrainPath, index=False)
isicVal.to_csv(isicValPath, index=False)
isicTest.to_csv(isicTestPath, index=False)
print(f'ISIC official train inventory: {len(isicTrain):,}; official test inventory: {len(isicTest):,}')
print('ISIC official test is reserved exclusively for zero-shot domain-shift evaluation.')

ISIC official train inventory: 22,215; official test inventory: 5,336
ISIC official test is reserved exclusively for zero-shot domain-shift evaluation.


In [7]:
trainValRepeat, testRepeat = chooseHoldout(clinicalData, 0.15, RANDOM_SEED)
assert set(clinicalData.iloc[testRepeat]['imageId']) == set(clinicalTest['imageId'])

def classCounts(frame: pd.DataFrame) -> dict[str, int]:
    return {str(label): int(count) for label, count in frame['harmonizedLabel'].value_counts().sort_index().items()}

splitReport = {
    'randomSeed': RANDOM_SEED,
    'targetFractions': {'train': 0.70, 'val': 0.15, 'test': 0.15},
    'clinical': {
        splitName: {
            'imageCount': len(frame),
            'patientCount': int(frame['patientId'].nunique()),
            'classCounts': classCounts(frame),
        }
        for splitName, frame in splitFrames.items()
    },
    'patientOverlapCounts': patientOverlap,
    'confirmedDuplicateOverlapCounts': duplicateOverlap,
    'patientLeakageDetected': any(patientOverlap.values()),
    'confirmedDuplicateLeakageDetected': any(duplicateOverlap.values()),
    'isic': {
        'labelsAvailable': isicLabelsAvailable,
        'officialTrainInventoryCount': len(isicTrain),
        'officialTestInventoryCount': len(isicTest),
        'supervisedSplitReady': True,
        'leakageGuarantee': 'Official train/test image IDs are disjoint; patient IDs are unavailable, so patient-level cross-partition overlap cannot be assessed.',
    },
    'repeatabilityCheckPassed': True,
}
splitReportPath = processedRoot / 'splitReport.json'
splitReportPath.write_text(json.dumps(splitReport, indent=2), encoding='utf-8')
print(json.dumps(splitReport, indent=2))

{
  "randomSeed": 42,
  "targetFractions": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "clinical": {
    "train": {
      "imageCount": 1479,
      "patientCount": 889,
      "classCounts": {
        "AK": 531,
        "BCC": 606,
        "MEL": 35,
        "NV": 169,
        "SCC": 138
      }
    },
    "val": {
      "imageCount": 290,
      "patientCount": 180,
      "classCounts": {
        "AK": 96,
        "BCC": 115,
        "MEL": 10,
        "NV": 37,
        "SCC": 32
      }
    },
    "test": {
      "imageCount": 294,
      "patientCount": 177,
      "classCounts": {
        "AK": 103,
        "BCC": 124,
        "MEL": 7,
        "NV": 38,
        "SCC": 22
      }
    }
  },
  "patientOverlapCounts": {
    "train-val": 0,
    "train-test": 0,
    "val-test": 0
  },
  "confirmedDuplicateOverlapCounts": {
    "train-val": 0,
    "train-test": 0,
    "val-test": 0
  },
  "patientLeakageDetected": false,
  "confirmedDuplicateLeakageDetected": false,
  "isic